# Repaso Clases 01 a 05 - Data Science I

| Ejemplo | Temas |
|---------|-------|
| 1 | Python y NumPy - fundamentos |
| 2 | Pandas - carga de CSV real, limpieza, agrupacion |
| 3 | Limpieza avanzada, strings, KNN Imputer |
| 4 | Matplotlib + Seaborn - visualizaciones con el CSV |

> Todos los ejemplos usan el archivo **ventas.csv** que esta en la misma carpeta.

---
# Ejemplo 1: Python y NumPy

Repaso rapido de los bloques fundamentales del lenguaje que se usan en todo Data Science.

In [ ]:
# ============================================================
# TIPOS DE DATOS EN PYTHON
# ============================================================
# Python tiene tipos simples (int, float, str, bool) y estructurados
# (list, tuple, dict, set). En DS los mas usados son list y dict,
# porque son la base de como Pandas y NumPy reciben los datos.

# int / float
edad = 28          # int: entero, sin decimales
precio = 1299.99   # float: con decimales

# list: coleccion ORDENADA y MUTABLE, permite duplicados
# Tipico uso: columnas de un DataFrame antes de crearlo
ventas_semana = [4500, 3200, 5100, 4800, 6000, 5500, 3900]

# dict: pares clave-valor, acceso rapido por nombre
# Tipico uso: crear un DataFrame desde cero en ejemplos y pruebas
producto = {'nombre': 'Laptop', 'precio': 1299.99, 'stock': 15}

print('Lista ventas:', ventas_semana)
print('Maximo:', max(ventas_semana), '| Minimo:', min(ventas_semana))
print('Dict producto:', producto)
print('Precio:', producto['precio'])


In [ ]:
# ============================================================
# FUNCIONES Y LIST COMPREHENSIONS
# ============================================================
# Las funciones encapsulan logica que se va a reutilizar.
# En DS son esenciales para encapsular pasos de preprocesamiento.

def clasificar_venta(valor):
    """Convierte un valor numerico en categoria cualitativa."""
    if valor >= 5000:
        return 'Alta'
    elif valor >= 3500:
        return 'Media'
    else:
        return 'Baja'

# List comprehension: aplicar la funcion a cada elemento de la lista
# Es mas compacto que un for loop y mas legible en Python
categorias = [clasificar_venta(v) for v in ventas_semana]
print('Ventas:', ventas_semana)
print('Categorias:', categorias)


## NumPy - Por que no usar listas de Python?

Las listas de Python son flexibles pero lentas para calculo numerico:
- Cada elemento es un **objeto Python** con overhead de memoria
- Las operaciones requieren loops elemento a elemento

NumPy almacena los datos como **bloques de memoria fija** (igual que C), lo que permite
operaciones **vectorizadas**: se aplican a todos los elementos a la vez, sin loops visibles.
Para 1 millon de datos, NumPy puede ser 100x mas rapido que una lista.

In [ ]:
import numpy as np

# ============================================================
# ARRAYS Y OPERACIONES VECTORIZADAS
# ============================================================
ventas = np.array([4500, 3200, 5100, 4800, 6000, 5500, 3900], dtype=float)

# Las operaciones aritmeticas se aplican a TODOS los elementos sin loop
ventas_con_iva = ventas * 1.21
print('Ventas sin IVA:', ventas)
print('Ventas + IVA 21%:', ventas_con_iva.round(2))

# Comparaciones: retornan array booleano -> sirven para filtrar
dias_buenos = ventas > 4500
print('Dias con ventas > 4500:', ventas[dias_buenos])

# Estadisticas descriptivas: siempre el punto de partida antes de modelar
print(f'\nMedia: {ventas.mean():.0f} | Mediana: {np.median(ventas):.0f}')
print(f'Desvio estandar: {ventas.std():.0f}')  # mide cuanto se dispersan los datos
print(f'Percentil 75: {np.percentile(ventas, 75):.0f}')  # el 75% esta por debajo de este valor


In [ ]:
# ============================================================
# ARRAYS 2D - Matrices
# ============================================================
# En Machine Learning los datos tienen forma de MATRIZ:
# filas = observaciones (muestras), columnas = variables (features)
# NumPy es el motor bajo el capo de Pandas y scikit-learn.

# Ventas de 4 productos en 3 regiones
ventas_2d = np.array([
    [4500, 3200, 5100, 4800],  # Region Norte
    [2800, 4100, 3600, 5200],  # Region Sur
    [6000, 5500, 4800, 7100],  # Region Centro
])
print('Forma (filas=regiones, cols=productos):', ventas_2d.shape)

# axis=1: opera a lo largo de columnas -> resultado por fila (por region)
# axis=0: opera a lo largo de filas   -> resultado por columna (por producto)
print('Total por region (axis=1):', ventas_2d.sum(axis=1))
print('Promedio por producto (axis=0):', ventas_2d.mean(axis=0).round(0))


---
# Ejemplo 2: Pandas - Carga y Analisis de Datos Reales

## Por que usar Pandas y no Excel o SQL?

- **vs Excel**: Pandas puede manejar millones de filas, automatizar procesos, y se integra con ML
- **vs SQL**: Pandas permite combinar operaciones en memoria sin servidor, con sintaxis Python
- **vs NumPy solo**: Pandas agrega etiquetas (nombres de columnas e indices), manejo de NaN
  nativo, y operaciones de alto nivel como groupby, merge, pivot

La estructura central es el **DataFrame**: una tabla con filas y columnas nombradas.
Cada columna es una **Series** (array 1D con etiqueta).

> El archivo **ventas.csv** tiene datos reales de ventas por Region, Producto y Mes.
> Tiene columnas: Region, Producto, Mes, Ventas, Satisfaccion_Cliente, Costo_Operativo

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# CARGA DEL CSV
# ============================================================
# pd.read_csv es la forma mas comun de ingestar datos en DS.
# El parametro sep detecta el separador (coma por defecto).
# encoding='utf-8' evita problemas con tildes y caracteres especiales.

df = pd.read_csv('ventas.csv', encoding='utf-8')

# SIEMPRE empezar con estos tres comandos al cargar un dataset nuevo:
print('SHAPE (filas, columnas):', df.shape)
print()
print('PRIMERAS FILAS:')
print(df.head())
print()
print('TIPOS DE DATOS:')
print(df.dtypes)


In [ ]:
# ============================================================
# INSPECCION ESTADISTICA
# ============================================================
# describe() es el primer vistazo a las distribuciones:
# - count: cuantos valores NO nulos (si difiere del total, hay NaN)
# - mean vs 50% (mediana): si difieren mucho, la distribucion esta sesgada
# - std: desvio estandar, mide la dispersion
# - min/max: detecta valores extremos que pueden ser outliers o errores
print(df.describe().round(2))
print()

# Variables categoricas: ver cuantos valores unicos hay
print('Regiones:', df['Region'].unique())
print('Productos:', df['Producto'].unique())
print('Cantidad de meses:', df['Mes'].nunique())


In [ ]:
# ============================================================
# FILTRADO Y SELECCION
# ============================================================
# loc: seleccion por ETIQUETA (nombre de indice o columna)
# iloc: seleccion por POSICION numerica (como indices de Python)
# Regla: usa loc cuando sabes el nombre, iloc cuando sabes la posicion

# Filtrado booleano: devuelve las filas que cumplen la condicion
ventas_altas = df[df['Ventas'] > 4000]
print(f'Filas con Ventas > 4000: {len(ventas_altas)} de {len(df)} totales')
print(ventas_altas.head(5))
print()

# Combinar condiciones con & (AND) | (OR)
# IMPORTANTE: cada condicion va entre parentesis
norte_alta = df[(df['Region'] == 'Norte') & (df['Ventas'] > 4000)]
print(f'Region Norte con Ventas > 4000: {len(norte_alta)} filas')


## GroupBy - Patron Split-Apply-Combine

**Por que usamos GroupBy?**

Cuando tenemos datos con categorias (Region, Producto, etc.) queremos comparar
el comportamiento entre grupos. GroupBy automatiza ese proceso:

1. **Split**: divide el DataFrame en grupos por Region (o cualquier columna)
2. **Apply**: aplica una funcion a cada grupo (sum, mean, max, etc.)
3. **Combine**: junta los resultados en un nuevo DataFrame

Es equivalente al `GROUP BY` de SQL y a las **tablas dinamicas** de Excel,
pero mucho mas flexible porque podemos aplicar cualquier funcion Python.

In [ ]:
# ============================================================
# GROUPBY - SPLIT-APPLY-COMBINE
# ============================================================

# agg() permite aplicar varias funciones a varias columnas a la vez
# Nombre de la columna resultado = nombre del argumento
resumen_region = df.groupby('Region').agg(
    ventas_total=('Ventas', 'sum'),
    ventas_promedio=('Ventas', 'mean'),
    satisfaccion_media=('Satisfaccion_Cliente', 'mean'),
    costo_total=('Costo_Operativo', 'sum'),
    registros=('Ventas', 'count'),
).round(2)

# Columna calculada: margen = ventas - costo
resumen_region['margen'] = resumen_region['ventas_total'] - resumen_region['costo_total']

print('Resumen por Region:')
print(resumen_region)
print()

# GroupBy con dos columnas: cruzar Region x Producto
cross = df.groupby(['Region', 'Producto'])['Ventas'].mean().round(0)
print('Ventas promedio por Region y Producto:')
print(cross)


In [ ]:
# ============================================================
# PIVOT TABLE Y NUEVA COLUMNA CALCULADA
# ============================================================
# pivot_table reorganiza el groupby en formato de tabla cruzada
# Es identico conceptualmente pero el resultado es mas visual

tabla = pd.pivot_table(
    df,
    values='Ventas',
    index='Region',
    columns='Producto',
    aggfunc='mean'
).round(0)
print('Ventas promedio (Region x Producto):')
print(tabla)
print()

# Agregar una columna nueva al DataFrame original
# El margen es informacion derivada que no estaba en el CSV
df['Margen'] = df['Ventas'] - df['Costo_Operativo']
df['Margen_Pct'] = (df['Margen'] / df['Ventas'] * 100).round(2)
print('Dataset con columnas calculadas:')
print(df[['Region', 'Producto', 'Mes', 'Ventas', 'Costo_Operativo', 'Margen', 'Margen_Pct']].head())


---
# Ejemplo 3: Limpieza de Datos - Strings, NaN y KNN Imputer

## Por que la limpieza es el paso mas critico?

Los datos reales **nunca** llegan listos para analizar. Los problemas mas comunes son:

- **NaN** (valores faltantes): pandas los representa como `NaN` (Not a Number)
- **Strings sucios**: espacios extra, mayusculas inconsistentes, caracteres raros
- **Outliers**: valores extremos que pueden ser errores de captura o casos reales excepcionales
- **Duplicados**: el mismo registro cargado dos veces

El principio **GIGO** (Garbage In, Garbage Out) dice que si entrenas un modelo con datos
sucios, el modelo va a aprender los errores. La limpieza es irreversible en cierto sentido:
lo que no limpias ahora, el modelo lo va a amplificar despues.

En este ejemplo simulamos datos sucios sobre el mismo dataset de ventas para practicar.

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# SIMULAR DATOS SUCIOS SOBRE EL CSV REAL
# ============================================================
# Cargamos el CSV limpio y le introducimos problemas tipicos
df_original = pd.read_csv('ventas.csv', encoding='utf-8')
df = df_original.copy()

# Problema 1: Region con espacios y mayusculas inconsistentes
# (tipico cuando distintas personas cargan el mismo formulario)
df.loc[[0, 5, 10], 'Region'] = '  norte '      # espacios extra
df.loc[[1, 6, 11], 'Region'] = 'NORTE'          # todo mayusculas
df.loc[[2, 7, 12], 'Region'] = 'sur '           # espacio al final

# Problema 2: NaN en columnas numericas (~10% de los datos)
np.random.seed(42)
idx_nan_ventas = np.random.choice(len(df), size=int(len(df)*0.10), replace=False)
idx_nan_sat    = np.random.choice(len(df), size=int(len(df)*0.08), replace=False)
df.loc[idx_nan_ventas, 'Ventas'] = np.nan
df.loc[idx_nan_sat, 'Satisfaccion_Cliente'] = np.nan

# Problema 3: duplicados
df = pd.concat([df, df.iloc[:3]], ignore_index=True)

print('Dataset con problemas:')
print(df.head(8))
print(f'\nForma: {df.shape}')
print('NaN por columna:')
print(df.isnull().sum())
print(f'Duplicados: {df.duplicated().sum()}')


## Limpieza de Strings con `.str`

Pandas tiene el accessor `.str` que aplica operaciones de texto
a una columna entera **sin necesidad de loops**.

Por que normalizar strings?
- `'Norte'`, `'NORTE'`, `'  norte '` son tres valores distintos para Python
- Un `groupby('Region')` los trataria como tres grupos diferentes
- Un modelo de ML los encodificaria como tres categorias distintas
- El resultado seria incorrecto aunque los datos sean conceptualmente iguales

In [ ]:
# ============================================================
# LIMPIEZA DE STRINGS
# ============================================================
df_clean = df.copy()

# strip() elimina espacios al inicio y al final
# title() capitaliza la primera letra de cada palabra
# La combinacion .strip().title() es el patron estandar de normalizacion
df_clean['Region'] = df_clean['Region'].str.strip().str.title()
df_clean['Producto'] = df_clean['Producto'].str.strip().str.title()

# Verificar: ahora todos los valores de Region son iguales
print('Valores unicos de Region ANTES:', df['Region'].unique())
print('Valores unicos de Region DESPUES:', df_clean['Region'].unique())

# Eliminar duplicados: keep='first' conserva la primera ocurrencia
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f'\nFilas antes: {len(df)} | Filas despues de drop_duplicates: {len(df_clean)}')


In [ ]:
# ============================================================
# TRATAMIENTO DE NaN - IMPUTACION SIMPLE
# ============================================================
# Antes de imputar, hay que ENTENDER el NaN:
# - Falta completamente al azar (MCAR): se puede imputar con media/mediana
# - Falta de forma sistematica (MAR/MNAR): imputar puede introducir sesgo

print('NaN en dataset limpio (strings y duplicados ya resueltos):')
print(df_clean.isnull().sum())
print()

# Mediana para Ventas: mas robusta que la media cuando hay outliers
# (valores extremos no afectan la mediana, si afectan la media)
mediana_ventas = df_clean['Ventas'].median()
print(f'Mediana de Ventas: {mediana_ventas:.0f}')

# Imputacion: fillna reemplaza NaN con el valor especificado
df_clean['Ventas'] = df_clean['Ventas'].fillna(mediana_ventas)

# Satisfaccion: media razonable (no tiene valores extremos)
media_sat = df_clean['Satisfaccion_Cliente'].mean()
df_clean['Satisfaccion_Cliente'] = df_clean['Satisfaccion_Cliente'].fillna(media_sat)

print('NaN despues de imputacion simple:', df_clean.isnull().sum().sum())


## KNN Imputer - Imputacion inteligente

La imputacion con **media o mediana** trata todos los registros igual.
Pero si los datos tienen estructura (correlaciones entre variables),
podemos imputar de forma **mas precisa** usando registros similares.

**KNN Imputer** (K-Nearest Neighbors):
1. Para el registro con NaN, busca los K mas parecidos (vecinos)
2. Usa el promedio de los vecinos para imputar el valor faltante

**Ejemplo concreto**: si una fila tiene `Costo_Operativo=5000` y `Ventas=NaN`,
KNN busca filas con `Costo_Operativo` similar y usa sus `Ventas` para imputar.
Es mucho mas preciso que usar la media global de todas las ventas.

**Cuando usarlo**: cuando hay correlacion entre variables y los NaN son pocos (<20%).

In [ ]:
from sklearn.impute import KNNImputer

# ============================================================
# KNN IMPUTER - IMPUTACION BASADA EN SIMILITUD
# ============================================================
# Volvemos al dataset con NaN para demostrar KNN Imputer
df_knn = df_clean.copy()

# Reintroducimos NaN en Ventas para ver como KNN los imputa
np.random.seed(10)
idx_test = np.random.choice(len(df_knn), size=8, replace=False)
valores_reales = df_knn.loc[idx_test, 'Ventas'].copy()  # guardamos para comparar
df_knn.loc[idx_test, 'Ventas'] = np.nan

# Seleccionar solo columnas numericas para el imputer
# KNNImputer no puede trabajar con strings
cols_numericas = ['Ventas', 'Satisfaccion_Cliente', 'Costo_Operativo']
imputer = KNNImputer(n_neighbors=5)  # usa 5 registros similares

# fit_transform calcula las distancias y devuelve el array imputado
arr_imputado = imputer.fit_transform(df_knn[cols_numericas])
df_imputado = df_knn.copy()
df_imputado[cols_numericas] = arr_imputado

# Comparar: valor real vs imputado por KNN vs media global
media_global = df_clean['Ventas'].mean()
print('Comparacion de imputaciones:')
print(f'Media global para todos: {media_global:.0f}')
print()
print(f'{"Idx":>5} | {"Costo":>8} | {"Real":>8} | {"KNN":>8} | {"Error KNN":>10}')
print('-' * 52)
for i in idx_test[:6]:
    real = valores_reales[i]
    knn  = df_imputado.loc[i, 'Ventas']
    costo = df_knn.loc[i, 'Costo_Operativo']
    print(f'{i:>5} | {costo:>8.0f} | {real:>8.0f} | {knn:>8.0f} | {abs(real-knn):>10.0f}')


---
# Ejemplo 4: Visualizaciones con Matplotlib y Seaborn

## Por que visualizar y por que hay DOS librerias?

La visualizacion es obligatoria en Data Science porque:
- Los numeros solos ocultan patrones que el ojo detecta de inmediato
- El **Cuarteto de Anscombe** muestra 4 datasets con identicas estadisticas
  (media, varianza, correlacion) pero con formas completamente distintas
- Permite comunicar resultados a stakeholders no tecnicos

**Matplotlib** es la libreria base del ecosistema Python:
- Control total sobre cada elemento del grafico
- Muy verbosa: cada detalle requiere una linea de codigo
- Ideal cuando necesitas graficos muy personalizados

**Seaborn** esta construida sobre Matplotlib pero eleva el nivel de abstraccion:
- Graficos estadisticos complejos con una sola linea
- Integracion directa con DataFrames (pasa el df y el nombre de columna)
- Estilos mas prolijos por defecto
- Ideal para analisis exploratorio rapido (EDA)

> **Regla practica**: usa Seaborn para explorar y entender los datos,
> usa Matplotlib para afinar el grafico final antes de presentarlo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Cargar el CSV y calcular el margen como en el Ejemplo 2
df = pd.read_csv('ventas.csv', encoding='utf-8')
df['Margen'] = df['Ventas'] - df['Costo_Operativo']

# Preparar serie temporal: extraer el numero de mes del string '2023-01'
df['mes_num'] = df['Mes'].str.split('-').str[1].astype(int)

# Agregar ventas totales por Region y Mes (para el lineplot)
ts = df.groupby(['Region', 'mes_num'])['Ventas'].sum().reset_index()

print('Dataset listo para visualizar:')
print(df.head())
print(f'\nForma: {df.shape} | Regiones: {df["Region"].unique()}')


## Como se edita un grafico en Matplotlib

Cada grafico en Matplotlib tiene una jerarquia de objetos:
- `Figure` (`fig`): el contenedor externo, como la hoja de papel
- `Axes` (`ax`): el area del grafico con ejes X e Y

Para personalizar un grafico se usan metodos del objeto `ax`:
- `ax.set_title('titulo')`: agrega titulo
- `ax.set_xlabel('x')` / `ax.set_ylabel('y')`: etiqueta los ejes
- `ax.set_xlim(min, max)` / `ax.set_ylim(min, max)`: define rango de ejes
- `ax.legend()`: muestra la leyenda
- `ax.grid(True)`: activa la grilla
- `ax.tick_params(...)`: personaliza los ticks
- `ax.text(x, y, 'texto')`: agrega texto en coordenadas especificas

El parametro `figsize=(ancho, alto)` en pulgadas controla el tamano total.

In [ ]:
# ============================================================
# GRAFICO 1: LINEA TEMPORAL CON ANOTACIONES
# ============================================================
# El lineplot es ideal para mostrar TENDENCIAS en el tiempo.
# Si usaramos barras, el ojo se enfoca en el tamano de cada barra individual.
# La linea conecta los puntos y hace visible la tendencia y estacionalidad.

fig, ax = plt.subplots(figsize=(12, 5))

# Colores y estilos por region
estilos = {
    'Norte':  {'color': 'steelblue', 'marker': 'o'},
    'Sur':    {'color': 'tomato',    'marker': 's'},
    'Centro': {'color': 'seagreen',  'marker': '^'},
}
meses_nombres = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

for region, estilo in estilos.items():
    datos_reg = ts[ts['Region'] == region].sort_values('mes_num')
    ax.plot(
        datos_reg['mes_num'],
        datos_reg['Ventas'] / 1000,
        marker=estilo['marker'],
        color=estilo['color'],
        linewidth=2,
        markersize=7,
        label=region
    )

# --- Personalizacion del grafico ---
ax.set_title('Evolucion de Ventas Mensuales por Region (2023)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mes', fontsize=12)
ax.set_ylabel('Ventas (miles $)', fontsize=12)

# Cambiar los numeros 1-12 por nombres de meses en el eje X
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_nombres, rotation=45, ha='right')

ax.legend(title='Region', fontsize=11)   # leyenda con titulo
ax.grid(True, linestyle='--', alpha=0.4) # grilla suave
ax.spines['top'].set_visible(False)      # ocultar borde superior (mas limpio)
ax.spines['right'].set_visible(False)    # ocultar borde derecho

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# GRAFICO 2: BARRAS AGRUPADAS CON TEXTO ENCIMA
# ============================================================
# Las barras son ideales para COMPARAR CATEGORIAS.
# El ojo puede comparar alturas mucho mas facilmente que longitudes o areas.
# Agrupadas (grouped) permiten comparar la misma metrica en distintos grupos.

resumen = df.groupby(['Region', 'Producto'])['Ventas'].mean().unstack()
productos = resumen.columns.tolist()
regiones = resumen.index.tolist()

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(regiones))
ancho = 0.2  # ancho de cada barra
palette = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']  # colores del tema seaborn

for i, (prod, color) in enumerate(zip(productos, palette)):
    offset = (i - len(productos)/2 + 0.5) * ancho  # centrar el grupo
    barras = ax.bar(x + offset, resumen[prod] / 1000,
                    width=ancho, label=prod, color=color, edgecolor='white')
    # Anotar el valor encima de cada barra
    for barra in barras:
        altura = barra.get_height()
        ax.text(
            barra.get_x() + barra.get_width() / 2,  # x: centro de la barra
            altura + 0.05,                           # y: un poco arriba de la barra
            f'{altura:.1f}K',                        # texto formateado
            ha='center', va='bottom', fontsize=8, color='#333333'
        )

ax.set_title('Ventas Promedio por Region y Producto', fontsize=13, fontweight='bold')
ax.set_ylabel('Ventas promedio (miles $)')
ax.set_xticks(x)
ax.set_xticklabels(regiones, fontsize=11)
ax.legend(title='Producto', loc='upper right')
ax.set_ylim(0, resumen.values.max() / 1000 * 1.25)  # espacio para las etiquetas
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Seaborn para analisis estadistico

Mientras Matplotlib se enfoca en COMO dibujar, Seaborn se enfoca en QUE mostrar.

Los graficos mas utiles de Seaborn para EDA (analisis exploratorio):

- **histplot + kde**: muestra la distribucion de una variable continua.
  El KDE (Kernel Density Estimation) suaviza el histograma para mostrar la
  forma de la distribucion sin depender del numero de bins elegido.

- **boxplot**: en una sola imagen muestra mediana, Q1, Q3, rango y outliers.
  Ideal para comparar distribuciones entre grupos categoricos.

- **heatmap**: muestra la correlacion entre todas las variables numericas.
  Permite identificar rapidamente que variables estan relacionadas,
  lo cual es critico antes de construir modelos (para evitar multicolinealidad).

In [ ]:
# ============================================================
# GRAFICO 3: DISTRIBUCION CON HISTPLOT + KDE (SEABORN)
# ============================================================
# El histograma muestra cuantas veces cae cada valor en un rango.
# El KDE (curva suavizada) permite ver la forma sin importar los bins.
# hue='Region' colorea automaticamente por grupo, sin loop manual.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribucion de Ventas por Region
sns.histplot(
    data=df, x='Ventas', hue='Region',
    kde=True,       # agrega la curva de densidad suavizada
    bins=20,        # numero de intervalos
    alpha=0.45,     # transparencia para que se vean los solapamientos
    ax=axes[0]
)
axes[0].set_title('Distribucion de Ventas por Region')
axes[0].set_xlabel('Ventas ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(True, linestyle='--', alpha=0.3)

# Distribucion de Satisfaccion por Region
sns.histplot(
    data=df, x='Satisfaccion_Cliente', hue='Region',
    kde=True, bins=15, alpha=0.45, ax=axes[1]
)
axes[1].set_title('Distribucion de Satisfaccion por Region')
axes[1].set_xlabel('Satisfaccion (1-5)')
axes[1].set_ylabel('Frecuencia')
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Distribuciones del Dataset de Ventas', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# GRAFICO 4: BOXPLOT + VIOLINPLOT (SEABORN)
# ============================================================
# Boxplot: la caja va de Q1 a Q3 (50% central de los datos)
#          la linea del medio es la MEDIANA (no la media)
#          los bigotes llegan hasta 1.5 * IQR
#          los puntos fuera son OUTLIERS candidatos
# Violinplot: agrega la forma de la distribucion a los lados (KDE rotado)
#             si la 'panza' esta arriba, la mayoria de datos esta arriba

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot: bueno para comparar medianas y dispersiones entre grupos
sns.boxplot(
    data=df, x='Producto', y='Ventas',
    hue='Region',   # sub-grupo por Region dentro de cada Producto
    palette='Set2', # esquema de colores de Seaborn
    ax=axes[0]
)
axes[0].set_title('Boxplot: Ventas por Producto y Region')
axes[0].set_xlabel('Producto')
axes[0].set_ylabel('Ventas ($)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, axis='y', linestyle='--', alpha=0.4)

# Violinplot: bueno cuando quieres ver la FORMA de la distribucion
# split=False muestra la distribucion completa por Region
sns.violinplot(
    data=df, x='Region', y='Ventas',
    palette='Set3', inner='quartile',  # inner='quartile' muestra las lineas de cuartiles
    ax=axes[1]
)
axes[1].set_title('Violinplot: Distribucion de Ventas por Region')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Ventas ($)')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# GRAFICO 5: HEATMAP DE CORRELACIONES
# ============================================================
# La CORRELACION de Pearson mide la relacion LINEAL entre dos variables.
# Valor +1: perfectamente correlacionadas (suben juntas)
# Valor -1: correlacion negativa perfecta (una sube, la otra baja)
# Valor  0: no hay relacion lineal (pueden tener relacion no lineal igual)

# Por que es importante antes de modelar?
# - Variables muy correlacionadas entre si causan MULTICOLINEALIDAD en regresion
# - Una variable muy correlacionada con el TARGET es un buen predictor
# - Variables correlacionadas con el target Y entre si -> elegir solo una

cols_num = ['Ventas', 'Satisfaccion_Cliente', 'Costo_Operativo', 'Margen']
corr = df[cols_num].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True,       # muestra el valor numerico en cada celda
    fmt='.2f',        # dos decimales
    cmap='coolwarm',  # azul=negativo, rojo=positivo, blanco=cero
    center=0,         # 0 es el centro de la escala de colores
    vmin=-1, vmax=1,  # limites fijos para poder comparar entre datasets
    square=True,      # celdas cuadradas
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlacion entre Variables Numericas', fontsize=13, pad=12)
plt.tight_layout()
plt.show()
print('Interpretacion:')
print(f'  Ventas vs Costo_Operativo: {corr.loc["Ventas","Costo_Operativo"]:.2f} -> relacion ?')
print(f'  Ventas vs Margen:          {corr.loc["Ventas","Margen"]:.2f} -> relacion ?')
print(f'  Ventas vs Satisfaccion:    {corr.loc["Ventas","Satisfaccion_Cliente"]:.2f} -> relacion ?')


In [ ]:
# ============================================================
# GRAFICO 6: SCATTER + REGRESION - Dos variables a la vez
# ============================================================
# El scatter plot muestra la relacion entre DOS variables continuas.
# Cada punto es un registro. Se puede agregar una tercera dimension con color (hue).
# regplot de Seaborn agrega automaticamente la linea de regresion con IC al 95%.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# scatter 1: Ventas vs Costo_Operativo coloreado por Region
sns.scatterplot(
    data=df, x='Costo_Operativo', y='Ventas',
    hue='Region', style='Producto',  # color=Region, forma=Producto
    alpha=0.7, s=60,                  # transparencia y tamano de puntos
    ax=axes[0]
)
axes[0].set_title('Ventas vs Costo Operativo')
axes[0].set_xlabel('Costo Operativo ($)')
axes[0].set_ylabel('Ventas ($)')
axes[0].grid(True, linestyle='--', alpha=0.3)

# regplot: scatter + linea de regresion + banda de confianza al 95%
# La banda gris muestra el INTERVALO DE CONFIANZA de la prediccion
# Si la banda es angosta, la relacion lineal es solida
sns.regplot(
    data=df, x='Satisfaccion_Cliente', y='Ventas',
    scatter_kws={'alpha': 0.5, 's': 30, 'color': 'steelblue'},
    line_kws={'color': 'red', 'linewidth': 2},
    ci=95,  # intervalo de confianza del 95% para la linea de regresion
    ax=axes[1]
)
axes[1].set_title('Ventas vs Satisfaccion (con regresion lineal)')
axes[1].set_xlabel('Satisfaccion del Cliente (1-5)')
axes[1].set_ylabel('Ventas ($)')
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()


---
# Resumen: Que aprendimos en Clases 01 a 05

| Clase | Concepto clave | Por que importa |
|-------|---------------|------------------|
| 01 | Pipeline Data Science | Entender las etapas evita errores de orden |
| 02 | Python + NumPy vectorizado | Calculo rapido sin loops en Python puro |
| 03 | Pandas DataFrame | Manipular tablas con nombres de columnas, no indices |
| 04 | NaN + KNN Imputer | Datos sucios producen modelos sucios (GIGO) |
| 05 | Matplotlib + Seaborn | Visualizar antes de modelar: el ojo detecta lo que los numeros ocultan |

**Flujo tipico de un proyecto Data Science (lo que practicamos aqui):**

```
1. Cargar datos        -> pd.read_csv()
2. Inspeccionar        -> df.head(), df.describe(), df.dtypes
3. Limpiar             -> strip(), fillna(), drop_duplicates(), KNNImputer
4. Transformar         -> columnas calculadas, groupby, pivot_table
5. Visualizar          -> lineplot, barras, histplot, boxplot, heatmap
6. (Clases 06-10)  -> estadistica inferencial, regresion, clasificacion, clustering
```